# Per-bit channel capacity

Single-read bit-level rate of the Babbage decoder, treated as a binary-symmetric-erasure channel (BSEC).

Pipeline: barcode02 reads -> HMM Viterbi -> 32 decoded bits per read -> compare to ground-truth bits of the predicted constant.

Bits are coded `{0, 1, -1}` where `-1` is an erasure (the alignment took a delete state or the transition was off-alphabet).

In [ ]:
import os
import sys
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# locate repo root robustly (notebook lives in analysis/channel_capacity/)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'data' / 'babbage.fasta').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'analysis'))

from shared import DataLoader, GROUND_TRUTH_BITS, BARCODE_CONDITIONS

# publication-friendly defaults
plt.rcParams.update({
    'font.sans-serif': ['Arial'],
    'font.family': 'sans-serif',
    'pdf.fonttype': 42,
    'font.size': 9,
})
PHI = (1 + 5 ** 0.5) / 2
WONG = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#F0E442']

## Load reads

Load barcode02 (PrimeStar GXL, 25c) which the same population used in the HP-level channel matrix. The full pass takes ~10 min on first call (FASTQ -> Viterbi against 17 HMMs); the `DataLoader` caches in memory so re-running cells is instant.

In [ ]:
loader = DataLoader(quick=False)
data = loader.load('merged', 'barcode02')
print(f'{len(data.reads):,} reads loaded')

## Compare each read's bits to ground truth

For every read that passed the filter and got a predicted constant, compare its 32 decoded bits to the ground-truth bits of that constant. Accumulate per-position counts of correct, flipped, and erased bits, plus a 2x3 joint distribution P(X, Y) where X is the truth bit and Y ∈ {0, 1, erased}.

In [ ]:
n_total_per_pos = np.zeros(32, dtype=np.int64)
n_invalid_per_pos = np.zeros(32, dtype=np.int64)
n_flip_per_pos = np.zeros(32, dtype=np.int64)
n_correct_per_pos = np.zeros(32, dtype=np.int64)
joint = np.zeros((2, 3), dtype=np.int64)  # rows: X={0,1}; cols: Y={0,1,erased}

n_with_bits = 0
for r in data.reads:
    if not r.result.passed_filter or not r.result.predicted_constant or r.result.bits is None:
        continue
    if len(r.result.bits) != 32:
        continue
    truth = GROUND_TRUTH_BITS.get(r.result.predicted_constant)
    if truth is None:
        continue
    n_with_bits += 1
    for i in range(32):
        b = r.result.bits[i]
        t = truth[i]
        n_total_per_pos[i] += 1
        if b < 0:
            n_invalid_per_pos[i] += 1
            joint[t, 2] += 1
        elif b == t:
            n_correct_per_pos[i] += 1
            joint[t, t] += 1
        else:
            n_flip_per_pos[i] += 1
            joint[t, 1 - t] += 1

print(f'reads with 32-bit vectors and matching truth: {n_with_bits:,}')
print(f'total bit-positions evaluated: {n_total_per_pos.sum():,}')

## Aggregate statistics

In [ ]:
total_bits   = int(n_total_per_pos.sum())
total_invalid = int(n_invalid_per_pos.sum())
total_valid   = total_bits - total_invalid
total_flip    = int(n_flip_per_pos.sum())
total_correct = int(n_correct_per_pos.sum())

p_erasure       = total_invalid / total_bits
p_flip_valid    = total_flip / max(total_valid, 1)
p_total_error   = (total_flip + 0.5 * total_invalid) / total_bits  # erasures as random

print(f'total bit-positions:                 {total_bits:>14,}')
print(f'invalid (erasure):                   {total_invalid:>14,}  ({100*p_erasure:.4f} %)')
print(f'valid:                               {total_valid:>14,}')
print(f'  correct:                           {total_correct:>14,}')
print(f'  flipped:                           {total_flip:>14,}')
print(f'P(erasure):                          {p_erasure:.6f}')
print(f'P(flip | valid):                     {p_flip_valid:.6f}')
print(f'P(error | erasures-as-random):       {p_total_error:.6f}')

## Rate metrics

Four candidates with progressively more accurate channel models:

1. **BSC, valid only** — `1 − H(p_flip)`. Conditional on a valid bit, this is the BSC capacity. Optimistic: pretends erasures don't exist.
2. **BSC, erasures as random** — `1 − H(p_total)`. Pessimistic: treats erasures as 50/50 guesses, lumped into the flip rate.
3. **BSEC formula** — `(1 − ε)(1 − H(p_flip))`. Correct when erasures are independent of input.
4. **Empirical I(X;Y)** — computed directly from the 2×3 joint P(X, Y∈{0, 1, erased}). Doesn't assume the channel is symmetric in the flip rates.

In [ ]:
def H_bin(p):
    if p <= 0 or p >= 1:
        return 0.0
    return -p * math.log2(p) - (1 - p) * math.log2(1 - p)

cap_bsc_valid_only      = 1 - H_bin(p_flip_valid)
cap_bsc_erasure_random  = 1 - H_bin(p_total_error)
cap_bsec                = (1 - p_erasure) * (1 - H_bin(p_flip_valid))

# empirical I(X;Y) from the joint counts
joint_p = joint / joint.sum()
p_x = joint_p.sum(axis=1, keepdims=True)
p_y = joint_p.sum(axis=0, keepdims=True)
mi_empirical = 0.0
for x in range(2):
    for y in range(3):
        pxy = joint_p[x, y]
        if pxy > 0:
            mi_empirical += pxy * math.log2(pxy / (p_x[x, 0] * p_y[0, y]))

print(f'1 − H(p_flip|valid)         = 1 − H({p_flip_valid:.4f})   = {cap_bsc_valid_only:.6f} bits/bit')
print(f'1 − H(p_total error)        = 1 − H({p_total_error:.4f})   = {cap_bsc_erasure_random:.6f} bits/bit')
print(f'BSEC: (1−ε)(1−H(p_flip))    = (1 − {p_erasure:.4f})·(1 − H({p_flip_valid:.4f}))')
print(f'                            = {cap_bsec:.6f} bits/bit')
print(f'empirical I(X; Y)           = {mi_empirical:.6f} bits')

## Per-position breakdown

Different bit positions correspond to different transitions in the codeword. The sign-bit position (0) is known to be the noisiest in this codec because of the asymmetric primer flank.

In [ ]:
p_invalid_pos    = n_invalid_per_pos / np.maximum(n_total_per_pos, 1)
p_flip_pos       = n_flip_per_pos / np.maximum(n_correct_per_pos + n_flip_per_pos, 1)
cap_bsec_per_pos = (1 - p_invalid_pos) * np.array([1 - H_bin(p) for p in p_flip_pos])

print(f'{"pos":>3}  {"n":>10}  {"erasure":>8}  {"flip|valid":>10}  {"1−H(flip)":>10}  {"BSEC":>8}')
for i in range(32):
    print(f'{i:>3}  {n_total_per_pos[i]:>10,}  {p_invalid_pos[i]:>8.4f}  {p_flip_pos[i]:>10.4f}  '
          f'{1-H_bin(p_flip_pos[i]):>10.4f}  {cap_bsec_per_pos[i]:>8.4f}')

print(f'\nposition-averaged: erasure={p_invalid_pos.mean():.4f}, flip={p_flip_pos.mean():.4f}, '
      f'BSEC={cap_bsec_per_pos.mean():.4f}')

In [ ]:
# per-position figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 7.2 / PHI / 2 * 1.6))
x = np.arange(32)

ax1.bar(x, p_flip_pos, width=0.8, color=WONG[1], label='P(flip | valid)')
ax1.bar(x, p_invalid_pos, width=0.8, bottom=p_flip_pos, color=WONG[5], label='P(erasure)')
ax1.set_xlabel('Bit position')
ax1.set_ylabel('Per-bit error rate')
ax1.set_title('Error decomposition by position')
ax1.legend(frameon=False)
for s in ('top', 'right'):
    ax1.spines[s].set_visible(False)

ax2.plot(x, cap_bsec_per_pos, '-o', markersize=3, color=WONG[0], label='BSEC mutual info')
ax2.axhline(cap_bsec, color=WONG[1], linestyle='--', linewidth=0.8,
            label=f'aggregate = {cap_bsec:.3f}')
ax2.set_xlabel('Bit position')
ax2.set_ylabel('Bits per bit')
ax2.set_title('Per-position BSEC mutual information')
ax2.set_ylim(0, 1.02)
ax2.legend(frameon=False, loc='lower right')
for s in ('top', 'right'):
    ax2.spines[s].set_visible(False)

fig.tight_layout()
plt.show()